In [6]:
import sys
sys.path.insert(0, "../../lib") 
from neurality import Neurality as N # type: ignore

text = """
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.
Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."
Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
""".strip().replace("\n", " ")

# Byte-level BPE: mulai dari 256 byte, lalu gabungkan pasangan terpopuler.
# Vocabulary kecil karena corpus contoh ini hanya satu paragraf.
class BytePairTokenizer:
    def __init__(self):
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = []

    @staticmethod
    def merge(tokens, pair, new_id):
        result = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
                result.append(new_id)
                i += 2
            else:
                result.append(tokens[i])
                i += 1
        return result

    def train(self, text, vocab_size=288):
        self.__init__()
        tokens = list(text.encode("utf-8"))
        while len(self.vocab) < vocab_size:
            counts = {}
            for pair in zip(tokens, tokens[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            if not counts:
                break
            pair = max(counts, key=counts.get)
            if counts[pair] < 2:
                break
            new_id = len(self.vocab)
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            self.merges.append((pair, new_id))
            tokens = self.merge(tokens, pair, new_id)
        return self

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for pair, new_id in self.merges:
            tokens = self.merge(tokens, pair, new_id)
        return tokens

    def decode(self, tokens):
        return b"".join(self.vocab[int(token)] for token in tokens).decode("utf-8", errors="replace")


tokenizer = BytePairTokenizer().train(text)
tokens = tokenizer.encode(text)
vocab_size = len(tokenizer.vocab)
context_length = len(tokens) - 1
assert tokenizer.decode(tokens) == text
print(f"Byte: {len(text.encode('utf-8'))}, token BPE: {len(tokens)}, vocabulary: {vocab_size}")

# Input dan target digeser satu token BPE.
inputs = [tokens[:-1]]
next_tokens = tokens[1:]

# CCE membutuhkan target one-hot
targets = [[
    [float(i == token) for i in range(vocab_size)]
    for token in next_tokens
]]

model = N()
model.embedding(vocab_size=vocab_size, embedding_size=16)
model.positional_embedding(max_length=context_length)

model.residual_start()
model.norm()
model.multihead_attention(num_heads=4, causal=True)
model.residual_end()

model.residual_start()
model.norm()
model.dense(64, N.Activations.gelu)
model.dense(16, N.Activations.linear)
model.residual_end()

model.norm()
model.dense(vocab_size, N.Activations.softmax)
model.optimizer(N.Optimizers.Adam(learning_rate=0.01))

# Latih untuk memprediksi token BPE berikutnya
for epoch in range(300):
    prediction = model.forward(inputs)
    loss = N.Loss.loss_cce(prediction, targets)
    model.backward(targets, N.Gradients.gradient_cce)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}: loss = {loss:.4f}")



Byte: 699, token BPE: 415, vocabulary: 288
Epoch 50: loss = 0.1289
Epoch 100: loss = 0.0105
Epoch 150: loss = 0.0055
Epoch 200: loss = 0.0035
Epoch 250: loss = 0.0025
Epoch 300: loss = 0.0018


In [9]:
# Generasi autoregresif: prediksi satu token BPE pada setiap langkah.
prompt = "Little girl"
generated = tokenizer.encode(prompt)

for _ in range(100):
    # Batasi konteks agar tidak melampaui kapasitas positional embedding.
    context = generated[-context_length:]
    probabilities = model.forward([context])[0][-1]
    next_id = max(range(vocab_size), key=lambda i: probabilities[i])
    generated.append(next_id)

print("Hasil:", tokenizer.decode(generated))


Hasil: Little girli. nit s s f wlnam the Lily f witd a s f her r fdlShe knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she 


In [10]:
print("Token prompt:", tokenizer.encode(prompt))
print("Potongan token:", [tokenizer.vocab[i] for i in tokenizer.encode(prompt)])
print("Hasil:", repr(tokenizer.decode(generated)))


Token prompt: [76, 266, 116, 275, 103, 276, 108]
Potongan token: [b'L', b'it', b't', b'le ', b'g', b'ir', b'l']
Hasil: 'Little girli. nit s s f wlnam the Lily f witd a s f her r fdlShe knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she '
